# Diving into Pinecone

This notebook uses the latest versions of the Pinecone library (version 5.0.0 or later).

Note: Older versions of the API used pinecone-client instead of pinecone. Make sure to upgrade using the **pinecone** package name instead of **pinecone-client**.

In [ ]:
pip install -q pinecone

In [ ]:
# pip install pinecone --upgrade -q

In [ ]:
# pip uninstall -y pinecone-client -q

In [2]:
pip show pinecone

Name: pinecone
Version: 6.0.1
Summary: Pinecone client and SDK
Home-page: 
Author: Pinecone Systems, Inc.
Author-email: support@pinecone.io
License: Apache-2.0
Location: /home/dda/projects/python/AI_Notebooks_OpenAI_LangChain_Copilot_Gemini/myenv311/lib/python3.11/site-packages
Requires: certifi, pinecone-plugin-interface, python-dateutil, typing-extensions, urllib3
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Authenticating to Pinecone. 
# the API KEY is in .env
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [4]:
from pinecone import Pinecone, ServerlessSpec

# Initilizing and authenticating the pinecone client
pc = Pinecone()
# pc = Pinecone(api_key='YOUR_API_KEY')

# checking authentication
pc.list_indexes()

[
    {
        "name": "langchain",
        "metric": "cosine",
        "host": "langchain-0e53e0f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "deepseek",
        "metric": "cosine",
        "host": "deepseek-0e53e0f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    }
]

## Working with Pinecone Indexes

### Creating a Serverless Pinecone index

In [5]:
# starter free plan permits 1 project, up to 5 indexes, up to 100 namespaces per index
index_name = 'rag'

if index_name not in pc.list_indexes().names():
    print(f'Creating index {index_name}')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ) 
    )
    print('Index created! 😊')
else:
    print(f'Index {index_name} already exists!')


Creating index rag
Index created! 😊


### Listing all indexes

In [6]:
pc.list_indexes()

[
    {
        "name": "langchain",
        "metric": "cosine",
        "host": "langchain-0e53e0f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "deepseek",
        "metric": "cosine",
        "host": "deepseek-0e53e0f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": 

In [7]:
index_name = 'rag'
# getting a complete description of a specific index:
pc.describe_index(index_name)

{
    "name": "rag",
    "metric": "cosine",
    "host": "rag-0e53e0f.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [8]:
# getting a list with the index names 
pc.list_indexes().names()

['langchain', 'deepseek', 'rag']

### Deleting indexes

In [9]:
# deleting an index
index_name = 'langchain'
if index_name in pc.list_indexes().names():
    print(f'Deleting index {index_name} ... ')
    pc.delete_index(index_name)
    print('Done')
else:
    print(f'Index {index_name} does not exist!')

Deleting index langchain ... 
Done


In [10]:
index_name = 'rag'
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

### Working with Vectors

In [11]:
# inserting vectors
import random
vectors = [[random.random() for _ in range(1536)] for v in range(5)]
# print(vectors)
ids = list('abcde')

index_name = 'rag'
index = pc.Index(index_name)

index.upsert(vectors=zip(ids, vectors))

{'upserted_count': 5}

In [12]:
# updating vectors
index.upsert(vectors=[('c', [0.5] * 1536)])

{'upserted_count': 1}

In [13]:
# fetching vectors
# index = pc.Index(index_name)
index.fetch(ids=['c', 'd'])

FetchResponse(namespace='', vectors={}, usage={'read_units': 1})

In [14]:
# deleting vectors
index.delete(ids=['b', 'c'])

{}

In [15]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [16]:
# querying a non-existing vector returns an empty vector
index.fetch(ids=['x']) 

FetchResponse(namespace='', vectors={}, usage={'read_units': 1})

In [17]:
# querying vectors
query_vector = [random.random() for _ in range(1536)]

In [18]:
index.query(
    vector=query_vector,
    top_k=3,
    include_values=False
)

{'matches': [], 'namespace': '', 'usage': {'read_units': 1}}

### Namespaces

In [19]:
# index.describe_index_stats()
index = pc.Index('rag')

import random
vectors = [[random.random() for _ in range(1536)] for v in range(5)]
ids = list('abcde')
index.upsert(vectors=zip(ids, vectors))

{'upserted_count': 5}

In [20]:
# partition the index into namespaces
# creating a new namespace
vectors = [[random.random() for _ in range(1536)] for v in range(3)]
ids = list('xyz')
index.upsert(vectors=zip(ids, vectors), namespace='first-namespace')

{'upserted_count': 3}

In [21]:
vectors = [[random.random() for _ in range(1536)] for v in range(2)]
ids = list('qp')
index.upsert(vectors=zip(ids, vectors), namespace='second-namespace')

{'upserted_count': 2}

In [22]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [23]:
index.fetch(ids=['x'])

FetchResponse(namespace='', vectors={}, usage={'read_units': 1})

In [24]:
index.fetch(ids=['x'], namespace='first-namespace')


FetchResponse(namespace='first-namespace', vectors={}, usage={'read_units': 1})

In [25]:
index.delete(ids=['x'], namespace='first-namespace')

{}

In [26]:
index.delete(delete_all=True, namespace='first-namespace')

{}

In [27]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}